# Scraping: récupération de la page d'accueil de la RTBF

Dans ce notebook, nous créons un robot qui va ouvrir la page d'accueil du site du journal [La RTBF](https://www.rtbf.be/) et récupérer le titre de tous les articles du jour et les stocker dans un fichier csv.

## Imports

In [1]:
import os
import re
import time
import requests
from bs4 import BeautifulSoup
import pandas as pd


In [3]:
from urllib.parse import urljoin

## Récupération de tous les articles de la page d'accueil



In [4]:

headers = {'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_10_1) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/39.0.2171.95 Safari/537.36'}

root_url = "https://max.de.wilde.web.ulb.be/camille/"
response = requests.get(root_url, headers=headers, timeout=30)
response.raise_for_status()
soup = BeautifulSoup(response.text, 'html.parser')

pdf_urls = []
for link in soup.find_all("a", href=True):
    href = link["href"]
    if href.lower().endswith(".pdf"):
        pdf_urls.append(urljoin(root_url, href))

print(len(pdf_urls))  # doit afficher 51
print(pdf_urls[:3])

51
['https://max.de.wilde.web.ulb.be/camille/KB_JB230_1892-08-07_01-0003.pdf', 'https://max.de.wilde.web.ulb.be/camille/KB_JB427_1920-01-10_01-00004.pdf', 'https://max.de.wilde.web.ulb.be/camille/KB_JB555_1836-02-08_01-00002.pdf']


In [8]:
# Affichage du nombre de pdf  récupérés
len(pdf_urls)

51

In [ ]:
# Affichage des 10 premières pdf
pdf_urls[:10]

['https://max.de.wilde.web.ulb.be/camille/KB_JB230_1892-08-07_01-0003.pdf',
 'https://max.de.wilde.web.ulb.be/camille/KB_JB427_1920-01-10_01-00004.pdf',
 'https://max.de.wilde.web.ulb.be/camille/KB_JB555_1836-02-08_01-00002.pdf',
 'https://max.de.wilde.web.ulb.be/camille/KB_JB638_1860-05-21_01-00002.pdf',
 'https://max.de.wilde.web.ulb.be/camille/KB_JB773_1918-11-30_01-00002.pdf',
 'https://max.de.wilde.web.ulb.be/camille/KB_JB838_1887-12-28_01-00003.pdf',
 'https://max.de.wilde.web.ulb.be/camille/KB_JB230_1903-10-16_01-0002.pdf',
 'https://max.de.wilde.web.ulb.be/camille/KB_JB427_1933-01-04_01-00003.pdf',
 'https://max.de.wilde.web.ulb.be/camille/KB_JB555_1899-01-19_01-00003.pdf',
 'https://max.de.wilde.web.ulb.be/camille/KB_JB638_1902-12-20_01-00002.pdf']

## Création d'un dataframe avec les liens et les titres des articles


In [5]:
df = pd.DataFrame(articles, columns=['link', 'title'])
df

,link,title
0,https://www.rtbf.behttps://www.rtbf.be/short/g...,
1,https://www.rtbf.behttps://www.rtbf.be/short/s...,
2,https://www.rtbf.behttps://www.rtbf.be/short/m...,
3,https://www.rtbf.behttps://www.rtbf.be/short/f...,
4,https://www.rtbf.behttps://www.rtbf.be/short/c...,
...,...,...
87,https://www.rtbf.behttps://auvio.rtbf.be/media...,Le 19h30
88,https://www.rtbf.behttps://auvio.rtbf.be/media...,"Salem, capitale mondiale d'halloween"
89,https://www.rtbf.behttps://auvio.rtbf.be/media...,En vrai de vrai
90,https://www.rtbf.behttps://auvio.rtbf.be/media...,Demain nous appartient


In [8]:
# Sauvegarde du dataframe dans un fichier csv
df.to_csv(f"../data/rtbf_{time.strftime('%Y%m%d')}.csv", index=False)

OSError: Cannot save file into a non-existent directory: '../data'

## Téléchargement d'un article et affichage du texte

In [7]:
# Récupération du premier lien du dataframe
article_url = df.iloc[0][0]

# Ouverture de la page
response = requests.get(article_url, headers=headers)
soup = BeautifulSoup(response.content, 'html.parser')
# Parsing du contenu
title = soup.find("h1").get_text(separator=' ')

paragraphs = []
for paragraph in soup.find_all("p", attrs={"class": None}):
    paragraphs.append(paragraph.get_text(separator=' ').strip())

content = "\n".join(paragraphs)
print(title)
print("==================================")
print(content)

KeyError: 0

### Nettoyage du texte à l'aide d'expressions régulières

In [ ]:
# Suppression de tout ce qui se trouve entre parenthèses
clean_content = re.sub("\([^\)]+\)", " ", content)
# Suppression des espaces multiples
clean_content = re.sub("\s+", " ", clean_content)

print(content)
print("==================================")
print(clean_content)


### Création d'un fichier avec le contenu de l'article


In [ ]:
with open("../data/rtbf_example.txt", "w") as writer:
    writer.write(f"{title}\n\n{clean_content}")

## Pour en savoir plus

- Le web scraping avec Python: https://realpython.com/beautiful-soup-web-scraper-python/
- Tutoriel sur les expressions régulières: https://www.w3schools.com/python/python_regex.asp